# Scrape CampusNest tutors and listings (Playwright async)

Target site: https://campusnest.streamlit.app/


In [ ]:
!pip -q install playwright pandas
!playwright install chromium

In [ ]:
import re
import pandas as pd
from urllib.parse import urljoin
from playwright.async_api import async_playwright

BASE_URL = "https://campusnest.streamlit.app/"


In [ ]:
async def discover_internal_pages(page, base_url: str) -> dict:
    await page.goto(base_url, wait_until="networkidle")
    await page.wait_for_timeout(1000)
    found = {}
    links = page.locator("a")
    n = await links.count()
    for i in range(n):
        a = links.nth(i)
        href = await a.get_attribute("href")
        label = (await a.inner_text() or "").strip()
        if not href or not label:
            continue
        abs_url = urljoin(base_url, href)
        if abs_url.startswith(base_url):
            found[label] = abs_url
    return found

def pick_page_url(pages: dict, contains: str) -> str:
    contains_low = contains.lower()
    for label, url in pages.items():
        if contains_low in label.lower():
            return url
    raise RuntimeError(f"Could not find page containing '{contains}'. Found: {list(pages.keys())[:20]}")

async def set_page_number(page, value: int):
    # targets the Streamlit number input labeled 'Page'
    label = page.get_by_text("Page", exact=True)
    container = label.locator("xpath=ancestor::div[1]")
    inp = container.locator("input[type='number']")
    if await inp.count() == 0:
        inp = page.locator("input[type='number']").first
    await inp.click()
    await inp.fill(str(value))
    await inp.press("Enter")
    await page.wait_for_timeout(800)

async def extract_card_texts(page) -> list[str]:
    cards = page.locator("div.sh-card")
    out = []
    n = await cards.count()
    for i in range(n):
        txt = (await cards.nth(i).inner_text()).strip()
        if txt:
            out.append(txt)
    return out

def parse_tutor_card(card_text: str) -> dict:
    lines = [ln.strip() for ln in card_text.splitlines() if ln.strip()]
    title = lines[0]
    m = re.match(r"^(?P<name>.+?)\s*·\s*€(?P<price>\d+)\/h$", title)
    if not m:
        m = re.match(r"^(?P<name>.+?)\s*·\s*€(?P<price>\d+)", title)
    name = m.group("name").strip() if m else title
    price = int(m.group("price")) if m and m.groupdict().get("price") else None

    headline = None
    subjects = None
    languages = None
    tutor_id = None
    for ln in lines[1:]:
        if ln.lower().startswith("subjects:"):
            subjects = ln.split(":", 1)[1].strip()
        elif ln.lower().startswith("languages:"):
            languages = ln.split(":", 1)[1].strip()
        elif ln.lower().startswith("id:"):
            tutor_id = ln.split(":", 1)[1].strip()
        else:
            if headline is None:
                headline = ln

    return {
        "id": tutor_id,
        "name": name,
        "price_eur_per_hour": price,
        "headline": headline,
        "subjects": subjects,
        "languages": languages,
        "raw": card_text,
    }

def parse_listing_card(card_text: str) -> dict:
    lines = [ln.strip() for ln in card_text.splitlines() if ln.strip()]
    title_line = lines[0]
    m = re.match(r"^(?P<title>.+?)\s*·\s*€(?P<rent>\d+)\/mo$", title_line)
    title = m.group("title").strip() if m else title_line
    rent = int(m.group("rent")) if m and m.groupdict().get("rent") else None

    listing_id = None
    listing_type = None
    area = None
    min_stay_months = None
    deposit_eur = None

    m2 = re.match(r"^(?P<type>.+?)\s+in\s+(?P<area>.+)$", title)
    if m2:
        listing_type = m2.group("type").strip()
        area = m2.group("area").strip()

    for ln in lines[1:]:
        if ln.lower().startswith("id:"):
            listing_id = ln.split(":", 1)[1].strip()
        if "min stay" in ln.lower():
            ms = re.search(r"min stay:\s*(\d+)\s*months", ln, flags=re.I)
            if ms:
                min_stay_months = int(ms.group(1))
            dep = re.search(r"deposit:\s*€(\d+)", ln, flags=re.I)
            if dep:
                deposit_eur = int(dep.group(1))

    return {
        "id": listing_id,
        "title": title,
        "type": listing_type,
        "area": area,
        "rent_eur_per_month": rent,
        "min_stay_months": min_stay_months,
        "deposit_eur": deposit_eur,
        "raw": card_text,
    }


In [ ]:
async def scrape_tutors_and_listings():
    tutor_rows = []
    listing_rows = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        pages = await discover_internal_pages(page, BASE_URL)
        tutoring_url = pick_page_url(pages, "Tutoring")
        housing_url = pick_page_url(pages, "Student Housing")

        # Tutors
        await page.goto(tutoring_url, wait_until="networkidle")
        await page.wait_for_timeout(1000)
        seen = set()
        for page_num in range(1, 60):
            await set_page_number(page, page_num)
            cards = await extract_card_texts(page)
            if not cards:
                break
            new_this_page = 0
            for c in cards:
                row = parse_tutor_card(c)
                if row.get("id") and row["id"] not in seen:
                    tutor_rows.append(row)
                    seen.add(row["id"])
                    new_this_page += 1
            if new_this_page == 0:
                break

        # Listings
        await page.goto(housing_url, wait_until="networkidle")
        await page.wait_for_timeout(1000)
        seen = set()
        for page_num in range(1, 200):
            await set_page_number(page, page_num)
            cards = await extract_card_texts(page)
            if not cards:
                break
            new_this_page = 0
            for c in cards:
                row = parse_listing_card(c)
                if row.get("id") and row["id"] not in seen:
                    listing_rows.append(row)
                    seen.add(row["id"])
                    new_this_page += 1
            if new_this_page == 0:
                break

        await browser.close()

    tutors_df = pd.DataFrame(tutor_rows).drop_duplicates(subset=["id"]).reset_index(drop=True)
    listings_df = pd.DataFrame(listing_rows).drop_duplicates(subset=["id"]).reset_index(drop=True)

    tutors_df.to_csv("campusnest_tutors.csv", index=False)
    listings_df.to_csv("campusnest_listings.csv", index=False)

    return tutors_df, listings_df

tutors_df, listings_df = await scrape_tutors_and_listings()
tutors_df.head(), listings_df.head(), (len(tutors_df), len(listings_df))
